# Ch.10 — Transformers & Attention
**Track:** ML from Scratch · California Housing dataset (8 features treated as a sequence of tokens)

## Core Idea

The LSTM (Ch.6) solved the vanishing gradient problem but still processes tokens **sequentially** — step 1 must complete before step 2. Transformers discard recurrence entirely and replace it with **attention**: a single parallel operation that lets every position directly compare itself to every other position at once.

```
RNN: x₁ → x₂ → x₃ → ... → xT (sequential bottleneck)
Transformer: [x₁, x₂, x₃, ..., xT] (all positions in parallel)
 ↓
 Multi-Head Attention: every position attends to every other
```

**The key equations:**

Scaled dot-product attention:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Positional encoding (sinusoidal):

$$\text{PE}_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad \text{PE}_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

## Running Example

The real estate platform has 8 tabular features per district in the California Housing dataset. We treat each feature as a **token** — a sequence of length 8. This is architecturally unconventional but pedagogically perfect: no new dataset, and the resulting attention heatmap has an immediately interpretable meaning ("which feature is attending to which other feature?").

Feature tokens: `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`

In [ ]:
# TODO: Implement this cell
#  (Load data)
#
# Steps:
# 1. Process data
# 2. Load data
# 3. Fit the model -- call `StandardScaler()`
# 4. Compute `X_tokens` using `astype()`
# 5. Call `T()` to produce the result
#
# Hint:
#    scaler = StandardScaler(???)
#    X_scaled = scaler.fit_transform(???)
#    scaler.fit_transform(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── Load data ─────────────────────────────────────────────────────────────────
data = fetch_california_housing()
X_raw, y = data.data, data.target      # (20640, 8), (20640,)
feature_names = list(data.feature_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw) # scale before projecting to tokens

# Treat each of the 8 features as a token with 1 raw dimension
# Shape: (N, T=8, 1)
X_tokens = X_scaled[:, :, np.newaxis].astype(np.float32)

print(f"Samples: {X_tokens.shape[0]}")
print(f"Sequence length T (tokens): {X_tokens.shape[1]}")
print(f"Raw token dimension: {X_tokens.shape[2]}")
print(f"Feature names (tokens): {feature_names}")

## The Math — Part 1: Positional Encoding

Attention is **permutation-equivariant**: shuffle the tokens and the output shuffles identically. Without positional information, the model cannot distinguish "feature 0 is MedInc" from "feature 5 is AveOccup". We inject position by **adding** a fixed encoding matrix to the token embeddings.

In [ ]:
def positional_encoding(T, d_model):
    """
    TODO #2: Implement `positional_encoding()`.

    Steps:
    1. Define helper function `positional_encoding()`
    2. Compute `PE`
    3. Plot: encoding matrix heatmap
    4. Call `0()` to produce the result

    Hint:
    PE = np.zeros(???)

    Returns: PE
    """
    raise NotImplementedError("TODO: implement positional_encoding()")

## The Math — Part 2: Scaled Dot-Product Attention

For each token, we project the input into three roles — **Query** (what am I looking for?), **Key** (what do I offer?), **Value** (what do I contribute if selected?). The attention weights are dot-product similarities between queries and keys, scaled by $\sqrt{d_k}$ to prevent softmax saturation.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    TODO #3: Implement `scaled_dot_product_attention()`.

    Steps:
    1. Define helper function `scaled_dot_product_attention()`
    2. Process data
    3. Call `exp()` to produce the result
    4. Process data
    5. Compute `rng` using `default_rng()`
    6. Compute `x_demo` using `value()`
    7. Compute `WQ` using `normal()`
    8. Compute `Q`
    9. Call `0()` to produce the result

    Hint:
    exp_scores = np.exp(???)
    rng = np.random.default_rng(???)
    x_demo = PE.copy(???)
    WQ = rng.normal(???)

    Returns: output, weights
    """
    raise NotImplementedError("TODO: implement scaled_dot_product_attention()")

## The Killer Visual — Attention Heatmap

Each row is a **query token** (the feature asking "who should I attend to?"). Each column is a **key token** (the feature being attended to). High weight = the model learned a strong relationship between those two features.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Plot results -- call `Query()`
#
# Hint:
#    Row = Query(???)
#    Col = Key(???)
#    vmax = weights_enc.max(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(weights_enc, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=feature_names, yticklabels=feature_names,
            vmin=0, vmax=weights_enc.max())
plt.title('Attention Weights — which feature "attends to" which?\n'
          'Row = Query (I am this feature)  |  Col = Key (I am attending to this)')
plt.xlabel('Key (attended-to feature)'); plt.ylabel('Query (attending feature)')
plt.tight_layout()
plt.show()

## Encoder vs Decoder — One Mask Difference

| | Encoder | Decoder |
|---|---|---|
| Mask | None — every position sees every other | Causal mask — position t sees only ≤ t |
| Trained for | Representation, classification, embeddings | Token generation, LLMs |
| Examples | BERT, embedding models | GPT-4, Llama, Claude |

Two lines of code swap the architecture.

In [ ]:
# TODO: Implement this cell
#  (Causal mask (decoder))
#
# Steps:
# 1. Causal mask (decoder)
# 2. Call `scaled_dot_product_attention()` to produce the result
# 3. Side-by-side comparison
# 4. Call `heatmap()` to produce the result
# 5. Plot results -- call `suptitle()`
#
# Hint:
#    causal_mask = np.triu(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Causal mask (decoder) ──────────────────────────────────────────────────────
causal_mask = np.triu(np.full((T, T), -np.inf), k=1)  # upper triangle = -inf
print("Causal mask:\n", causal_mask)

output_dec, weights_dec = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

# ── Side-by-side comparison ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(weights_enc, ax=ax1, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=feature_names, yticklabels=feature_names, vmin=0)
ax1.set_title('ENCODER — full attention\n(every feature sees every other)', fontweight='bold')
ax1.set_xlabel('Key'); ax1.set_ylabel('Query')

sns.heatmap(weights_dec, ax=ax2, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=feature_names, yticklabels=feature_names, vmin=0)
ax2.set_title('DECODER — causal mask\n(each feature sees only itself and earlier features)', fontweight='bold')
ax2.set_xlabel('Key'); ax2.set_ylabel('Query')

plt.suptitle('One mask → two architectures → BERT vs GPT', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Multi-Head Attention

One set of Q, K, V learns one relationship pattern. Run H heads in parallel, each with its own projections (dimension `d_k = d_model / H`), then concatenate and project back.

One head might track income-location correlations. Another might track occupancy-room-count interactions. Each head specialises independently.

In [ ]:
def multi_head_attention(X, num_heads, d_model):
    """
    TODO #6: Implement `multi_head_attention()`.

    Steps:
    1. Define helper function `multi_head_attention()`
    2. Call `normal()` to produce the result
    3. Call `concatenate()` to produce the result
    4. Process data
    5. Plot results -- call `subplots()`
    6. Plot results -- call `suptitle()`

    Hint:
    rng = np.random.default_rng(???)
    WQ_h = rng.normal(???)
    WK_h = rng.normal(???)
    WV_h = rng.normal(???)

    Returns: output, all_weights
    """
    raise NotImplementedError("TODO: implement multi_head_attention()")

## Full Transformer Encoder in Keras

Now we wire together the full encoder: token projection → positional encoding → N encoder blocks (LayerNorm + Multi-Head Attention + residual + FFN + residual) → global average pool → regression head.

In [ ]:
def build_tabular_transformer(T=8, d_in=1, d_model=32, num_heads=4,
                               num_layers=2, ffn_dim=64, dropout=0.1):
    """
    TODO #7: Implement `build_tabular_transformer()`.

    Steps:
    1. Process data
    2. Call `set_seed()` to produce the result
    3. Define helper function `build_tabular_transformer()`
    4. Call `Dense()` to produce the result
    5. Call `encoding()` to produce the result
    6. Call `LayerNormalization()` to produce the result
    7. Call `Dense()` to produce the result
    8. Call `Model()` to produce the result
    9. Compute `model` using `summary()`

    Hint:
    inputs = keras.Input(???)
    x = layers.Dense(???)
    z = layers.LayerNormalization(???)
    z = layers.MultiHeadAttention(???)

    Returns: keras.Model(inputs, outputs, name='Ta...
    """
    raise NotImplementedError("TODO: implement build_tabular_transformer()")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Call `train_test_split()` to produce the result
# 3. Call `compile()` to produce the result
# 4. Compute `early_stop` using `EarlyStopping()`
# 5. Fit the model -- call `fit()`
# 6. Process data
#
# Hint:
#    optimizer = keras.optimizers.Adam(???)
#    early_stop = keras.callbacks.EarlyStopping(???)
#    history = model.fit(???)
#    model.fit(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_tokens, y.astype(np.float32), test_size=0.2, random_state=42)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=[keras.metrics.RootMeanSquaredError(name='rmse')])

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_tr, y_tr,
    epochs=100, batch_size=256,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0)

print(f"Stopped at epoch {len(history.history['loss'])}")

In [ ]:
# TODO: Implement this cell
#  (Training curves)
#
# Steps:
# 1. Training curves
# 2. Call `plot()` to produce the result
# 3. Plot results -- call `tight_layout()`
# 4. Compute `results` using `evaluate()`
#
# Hint:
#    results = model.evaluate(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Train loss (MSE)')
ax1.plot(history.history['val_loss'], label='Val loss (MSE)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE'); ax1.set_title('Loss curves')
ax1.legend()

ax2.plot(history.history['rmse'], label='Train RMSE')
ax2.plot(history.history['val_rmse'], label='Val RMSE')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('RMSE ($100k)'); ax2.set_title('RMSE curves')
ax2.legend()

plt.tight_layout(); plt.show()

# Final evaluation
results = model.evaluate(X_te, y_te, verbose=0)
print(f"\nTest MSE:  {results[0]:.4f}")
print(f"Test RMSE: {results[1]:.4f}  (in $100k units — so ${results[1]*100:.0f}k typical error)")

## Parameter Count: LSTM vs Transformer

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `lstm_model` using `Dense()`
# 2. Compute `tr_params` using `count_params()`
# 3. Process data
#
# Hint:
#    lstm_model = keras.Sequential(???)
#    tr_params = model.count_params(???)
#    lstm_params = lstm_model.count_params(???)
#    lstm_model.compile(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Build equivalent LSTM for comparison
lstm_model = keras.Sequential([
    keras.Input(shape=(T, 1)),
    layers.LSTM(32),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
], name='LSTM_baseline')
lstm_model.compile(optimizer='adam', loss='mse',
                   metrics=[keras.metrics.RootMeanSquaredError(name='rmse')])

tr_params = model.count_params()
lstm_params = lstm_model.count_params()

print("=" * 50)
print(f"  Transformer params : {tr_params:>8,}")
print(f"  LSTM params        : {lstm_params:>8,}")
print("=" * 50)
print()
print("Key difference — not parameters, but computation structure:")
print("  LSTM : 8 sequential steps  → cannot parallelise across tokens")
print("  Transformer : 1 parallel pass → full GPU utilisation at any sequence length")

## The Hyperparameter Dial

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Fit the model -- call `build_tabular_transformer()`
# 2. Compute `ds`
# 3. Plot results -- call `subplots()`
# 4. Compute `ax2` using `twinx()`
# 5. Plot results -- call `title()`
#
# Hint:
#    optimizer = keras.optimizers.Adam(???)
#    h = m.fit(???)
#    params = m.count_params(???)
#    ax1 = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Sweep d_model — the single most impactful dial for small transformers
results_sweep = []
for d in [8, 16, 32, 64]:
    m = build_tabular_transformer(d_model=d, num_heads=4 if d >= 16 else 2, ffn_dim=d*4)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse',
              metrics=[keras.metrics.RootMeanSquaredError(name='rmse')])
    h = m.fit(X_tr, y_tr, epochs=40, batch_size=256,
              validation_split=0.15, verbose=0)
    val_rmse = min(h.history['val_rmse'])
    params = m.count_params()
    results_sweep.append({'d_model': d, 'params': params, 'val_rmse': val_rmse})
    print(f"d_model={d:3d}  params={params:6,}  best val RMSE={val_rmse:.4f}")

# Plot
ds     = [r['d_model']  for r in results_sweep]
rmses  = [r['val_rmse'] for r in results_sweep]
params = [r['params']   for r in results_sweep]

fig, ax1 = plt.subplots(figsize=(8, 4))
color1 = 'steelblue'
ax1.plot(ds, rmses, 'o-', color=color1, label='Val RMSE')
ax1.set_xlabel('d_model'); ax1.set_ylabel('Val RMSE ($100k)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'coral'
ax2.plot(ds, params, 's--', color=color2, label='# Parameters')
ax2.set_ylabel('Parameter count', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('d_model sweep — accuracy vs parameter cost')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout(); plt.show()

## Exercises

1. **Change the mask.** In the encoder/decoder side-by-side cell, modify the causal mask to instead mask out only the **diagonal** (a token cannot attend to itself). Re-run the heatmap. Does the model still make sense? When would this be useful?

2. **Increase heads.** Change `num_heads` from 4 to 8 in `build_tabular_transformer`. What happens to `key_dim`? What constraint does this impose on `d_model`? Try `d_model=24, num_heads=8` — does it error? Why?

3. **Remove positional encoding.** Comment out the `x = x + pe[...]` line. Retrain and compare val RMSE. Does it matter for tabular data? Why would it matter more for natural language?

## Bridge

Ch.10 built the transformer encoder from first principles — attention, positional encoding, residuals, LayerNorm, and the one-mask difference between BERT and GPT. This is the architecture that every embedding model, every LLM, and every foundation model runs on.

The AI track's `RAGAndEmbeddings` note picks up exactly here: embedding models *are* transformer encoders trained with contrastive loss to produce sentence vectors you can compare by cosine similarity. The pooling step in those notes (mean-pool across token outputs) is exactly the `GlobalAveragePooling1D` layer at the end of this chapter's model.

> *One architecture. Three deployment patterns: encode for embeddings (BERT), generate autoregressively (GPT), or condition on a separate encoder (T5-style encoder-decoder).*